# 05 · 인간의 검증력: 시각화로 오류 잡기  (모듈 5)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amnotyoung/oda-ai-stats/blob/main/notebooks/05_human_verification.ipynb)

> AI가 만든 "그럴듯한 분석"을 **인간의 3대 무기 — 🖼️시각화 · 🧭도메인 지식 · 👥동료 회람**으로 검증한다.
> AI는 빠르지만 *보지 못하고, 맥락이 없고, 혼자* 일한다.

In [ ]:
# 한글 폰트 설정 — Colab 그래프의 한글이 깨지지 않도록 (처음 1회 약 20초)
import os, matplotlib.pyplot as plt, matplotlib.font_manager as fm
try:
    p = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
    if not os.path.exists(p):
        os.system("apt-get -qq install -y fonts-nanum > /dev/null 2>&1")
    fm.fontManager.addfont(p)
    plt.rc("font", family="NanumGothic")
except Exception:
    pass
plt.rc("axes", unicode_minus=False)

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
df = pd.read_csv("https://raw.githubusercontent.com/amnotyoung/oda-ai-stats/main/data/sample_crs.csv")

## 시나리오 — "보건 분야 ODA는 어떻게 변했나?"
AI에게 분석을 시켰더니 아래 표를 줬다. **표만 보면 완벽해 보인다.**

In [ ]:
health = df[df.SectorName == "Health"]
yearly = health.groupby("Year")["USD_Disbursement"].sum().round(1)
yearly  # 표: 깔끔해 보인다

### 🖼️ 무기 1 — **그려본다**
현업에선 데이터에 오류가 섞이기 쉽다. 한 대형사업이 **이중계상**됐다고 해보자(흔한 실수).
표로는 안 보이지만, **그리면** 바로 드러난다.

In [ ]:
# 흔한 오류를 모사: 2019년 한 대형 보건사업이 실수로 10배 중복 입력됐다고 가정
dirty = df.copy()
big = dirty[(dirty.SectorName == "Health") & (dirty.Year == 2019)].sort_values("USD_Disbursement").tail(1)
dup = pd.concat([big] * 9, ignore_index=True)        # 같은 사업 9번 더 추가(이중계상)
dirty = pd.concat([dirty, dup], ignore_index=True)

y_dirty = dirty[dirty.SectorName == "Health"].groupby("Year")["USD_Disbursement"].sum()
y_clean = df[df.SectorName == "Health"].groupby("Year")["USD_Disbursement"].sum()
plt.figure(figsize=(8, 4))
plt.plot(y_dirty.index, y_dirty.values, "o-", label="AI가 받은 데이터(오류 포함)")
plt.plot(y_clean.index, y_clean.values, "s--", label="실제(정상)")
plt.title("보건 ODA 연도별 추세 — 표는 속삭이지만 그림은 소리친다")
plt.legend(); plt.tight_layout(); plt.show()

→ **2019년 비정상 급등**이 한눈에. 표의 숫자만 봤다면 보고서에 그대로 들어갔을 오류다.

In [ ]:
# 범인 찾기: 같은 사업이 여러 번? 도메인 지식으로 '말이 되나' 확인
susp = dirty[(dirty.SectorName == "Health") & (dirty.Year == 2019)]
susp["ProjectTitle"].value_counts().head(3)   # 한 사업명이 비정상적으로 여러 번

### 🧭 무기 2 — **도메인 지식**
"이 수원국이 그 해 보건 최대 수혜국? 현장을 아는 사람은 *말이 안 된다*는 걸 안다."
AI에겐 현장의 사실(ground truth)이 없다. **여러분에겐 있다.**

### 👥 무기 3 — **동료 회람**
1장 요약을 그 사업/지역을 아는 **동료에게 회람**: "그 사업 2018년에 종료됐는데?"
한 사람(혹은 AI)이 놓친 걸 **여러 눈**이 잡는다.

---
✅ **결론**: AI가 분석을 *쉽게* 만들어도, *맞게* 만드는 건 인간이다.
시각화·도메인·회람 — 이 무기를 매번 쓰자. (체크리스트 → `handouts/verification_checklist.md`)